# StandAlone FBA using metabolism_redux

modified from original (Heena's) by Chris on 4/23/26

In [1]:
import pandas as pd
import os, dill
import numpy as np
import cvxpy as cp
import plotly.express as px

from ecoli.processes.metabolism_redux import NetworkFlowModel, FlowResult

os.chdir(os.path.expanduser('~/projects/SMS/vEcoli'))
# os.chdir(os.path.expanduser('~/projects/vEcoli'))  

%load_ext autoreload


In [2]:
def load_sim(
        folder_path:str,
):
    """ This function is meant to load an output of a simulation in timeseries form.
        Note: This is not designed for parquet output format.
    """
    # --- Load Sim ---
    output = np.load(folder_path + '0_output.npy',allow_pickle='TRUE').item()
    output = output['agents']['0']
    fba = output['listeners']['fba_results']
    bulk = pd.DataFrame(output['bulk'])
    f = open(folder_path + 'agent_steps.pkl', 'rb')
    agent = dill.load(f)
    f.close()

    metabolism = agent['ecoli-metabolism-redux']

    return fba, bulk, metabolism, output

# Define helper functions for standalone FBA

In [3]:
def get_subset_S(S, met_of_interest):
    S_met = S.loc[met_of_interest, :]
    S_met = S_met.loc[:, ~np.all(S_met == 0, axis=0)]
    return S_met, S_met.columns


def get_keys(dict, value):
    return [key for key in dict if np.any(np.isin(value, dict[key]))]


def test_NetworkFlowModel(objective_weights, metabolism, fba, bulk,
                          uptake_addition=set(),
                          uptake_removal=set(),
                          new_exchange_molecules=set(),
                          add_metabolite=None,
                          add_reaction=None,
                          remove_reaction=None,
                          add_kinetic=None,
                          add_homeostatic_demand=None,
                          solver_choice=cp.GLOP):
    """Standalone FBA against the metabolism_redux NetworkFlowModel.

    Note: `force_reaction` (hard-force flux) is not supported in metabolism_redux.
    Use `add_kinetic={'RXN-ID': v}` for a soft-force via the kinetic objective, which
    can be paired with `remove_reaction` for the opposing direction.
    """

    # --- Update Exchanges ---
    # NetworkFlowModel.set_up_exchanges requires uptakes ⊆ exchanges (it sizes
    # S_exch as len(exchanges) + len(uptakes), assuming uptakes are also
    # exchangeable). The redux process itself unions these before calling.
    uptake = set(metabolism.allowed_exchange_uptake) | uptake_addition
    uptake = uptake - uptake_removal
    exchange_molecules = set(metabolism.exchange_molecules) | new_exchange_molecules | uptake

    # --- Get whole-cell model outputs for FBA ---
    reaction_names = list(metabolism.reaction_names)
    metabolites = list(metabolism.metabolite_names)
    kinetic_reaction_ids = list(metabolism.kinetic_constraint_reactions)

    # Drop t=0 (process has not run yet — empty rows)
    # kinetic targets: 2-D [lower, target, upper] per reaction
    kin_target = np.array(fba["target_kinetic_fluxes"][1:]).mean(axis=0)         # (n_kin,)
    kin_bounds = np.array(fba["target_kinetic_bounds"][1:]).mean(axis=0)         # (n_kin, 2) = [lower, upper]
    kinetic = pd.DataFrame(
        np.column_stack([kin_bounds[:, 0], kin_target, kin_bounds[:, 1]]),
        index=kinetic_reaction_ids,
        columns=["lower", "target", "upper"],
    )

    # homeostatic dm/dt targets (counts/s, mean over post-t0 timesteps)
    homeostatic = pd.Series(
        np.array(fba["target_homeostatic_dmdt"][1:]).mean(axis=0),
        index=list(metabolism.homeostatic_metabolites),
    )

    # homeostatic metabolite counts — not in listener; pull from bulk via index
    homeostatic_counts = pd.Series(
        bulk.iloc[1:, metabolism.homeostatic_metabolite_idx].mean(axis=0).values,
        index=list(metabolism.homeostatic_metabolites),
    )
    # Concentrations (mmol/L) for the redux solve's `homeostatic_concs` denominator
    homeostatic_concs = homeostatic_counts * metabolism.counts_to_molar.asNumber()

    # Scalar mean maintenance flux (used as ngam_target — approximation; the
    # listener emits post-solve total maintenance, which includes gam*exch terms)
    maintenance = float(np.mean(fba["maintenance_target"][1:]))

    S_new = np.asarray(metabolism.stoichiometry.todense())  # redux stores csr_matrix; densify for perturbation math

    # --- Add/remove reactions, metabolites, kinetic constraints, homeostatic demands ---
    if add_metabolite is not None:
        for m in add_metabolite:
            if m not in metabolites:
                metabolites.append(m)
        S_new = np.concatenate(
            (S_new, np.zeros((len(add_metabolite), S_new.shape[1]))), axis=0
        )

    if add_reaction is not None:
        assert isinstance(add_reaction, dict)
        for r, s in add_reaction.items():
            assert isinstance(s, dict)
            if r not in reaction_names:
                reaction_names.append(r)
            new_reaction = np.zeros((S_new.shape[0], 1))
            for m, v in s.items():
                new_reaction[metabolites.index(m), 0] = v
            S_new = np.concatenate((S_new, new_reaction), axis=1)

    if add_kinetic is not None:
        # {"reaction_name": target_value} → encoded as [v, v, v] (hard target)
        assert isinstance(add_kinetic, dict)
        for r, v in add_kinetic.items():
            if r not in kinetic_reaction_ids:
                kinetic_reaction_ids.append(r)
            kinetic.loc[r] = [v, v, v]

    if remove_reaction is not None:
        for r in remove_reaction:
            r_idx = reaction_names.index(r)
            S_new = np.delete(S_new, r_idx, axis=1)
            reaction_names.remove(r)
            if r in kinetic_reaction_ids:
                kinetic_reaction_ids.remove(r)
                kinetic = kinetic.drop(index=r)

    if add_homeostatic_demand is not None:
        # NB: pre-existing semantics — adds to local Series but NOT to the
        # NetworkFlowModel's homeostatic_metabolites list, so will desync.
        # Carrying forward as-is; flag if exercised.
        assert isinstance(add_homeostatic_demand, list)
        for met in add_homeostatic_demand:
            homeostatic[met] = 100
            homeostatic_counts[met] = 1
            homeostatic_concs[met] = 1 * metabolism.counts_to_molar.asNumber()

    # --- Solve NetworkFlowModel ---
    model = NetworkFlowModel(
        stoich_arr=S_new,
        metabolites=metabolites,
        reactions=reaction_names,
        homeostatic_metabolites=list(metabolism.homeostatic_metabolites),
        kinetic_reactions=kinetic_reaction_ids,
        get_mass=metabolism.parameters["get_mass"],
        gam=metabolism.gam.asNumber(),
        # active_constraints_mask intentionally left None — perturbations may
        # change kinetic_reaction_ids length, invalidating the saved 416-mask
    )
    model.set_up_exchanges(exchanges=exchange_molecules, uptakes=uptake)
    solution: FlowResult = model.solve(
        homeostatic_concs=homeostatic_concs.values,
        homeostatic_dm_targets=homeostatic.values,
        ngam_target=maintenance,
        kinetic_targets=kinetic[["lower", "target", "upper"]].values,
        binary_kinetic_idx=None,
        objective_weights=objective_weights,
        upper_flux_bound=1_000_000_000,  # counts/s, not conc/s
        solver=solver_choice,
    )
    return solution.objective, solution.velocities, reaction_names, S_new, metabolites, kinetic


# Load Sim and run standalone FBA

In [4]:
folder_path = "out/metabolic_engineering/basal/metabolism_redux_metabolic_engineering_600_2026-04-23/"

fba, bulk, metabolism, output = load_sim(folder_path)


## Aside — explore the model surface


In [5]:
# Quick look at the model's structural surface.
print(f"Reactions:                {len(metabolism.reaction_names):>6,}")
print(f"Metabolites:              {len(metabolism.metabolite_names):>6,}")
print(f"Kinetically constrained:  {len(metabolism.kinetic_constraint_reactions):>6,}")
print(f"Homeostatic targets:      {len(metabolism.homeostatic_metabolites):>6,}")
print(f"Exchangeable molecules:   {len(metabolism.exchange_molecules):>6,}")
print(f"Allowed uptakes:          {len(metabolism.allowed_exchange_uptake):>6,}")

print(f"\nFirst 5 reaction names:   {metabolism.reaction_names[:5]}")
print(f"First 5 metabolite names: {metabolism.metabolite_names[:5]}")


Reactions:                 9,421
Metabolites:               6,149
Kinetically constrained:     416
Homeostatic targets:         172
Exchangeable molecules:       87
Allowed uptakes:              19

First 5 reaction names:   ['1-ACYLGLYCEROL-3-P-ACYLTRANSFER-RXN', '1.1.1.127-RXN', '1.1.1.127-RXN (reverse)', '1.1.1.215-RXN (reverse)', '1.1.1.251-RXN']
First 5 metabolite names: ['1-2-Diglycerides[c]', '1-AMINO-PROPAN-2-ONE-3-PHOSPHATE[c]', '1-Cys-Peroxiredoxin-L-cysteine[c]', '1-Cys-Peroxiredoxin-L-hydroxycysteine[c]', '1-KETO-2-METHYLVALERATE[c]']


In [6]:
# Kinetic constraints come from WCM enzyme expression × kinetic parameters at
# parca time — only a fraction of reactions carry them.
n_rxn = len(metabolism.reaction_names)
n_kin = len(metabolism.kinetic_constraint_reactions)
print(f"{n_kin} of {n_rxn} reactions ({n_kin / n_rxn:.1%}) carry kinetic constraints.")
print(f"\nFirst 10 kinetic-constrained reactions:")
for r in metabolism.kinetic_constraint_reactions[:10]:
    print(f"  {r}")


416 of 9421 reactions (4.4%) carry kinetic constraints.

First 10 kinetic-constrained reactions:
  1.1.1.39-RXN
  1.1.1.83-RXN
  1.13.11.16-RXN
  2.1.1.79-RXN-CPD-18361/S-ADENOSYLMETHIONINE//CPD-18373/ADENOSYL-HOMO-CYS/PROTON.67.
  2.1.1.79-RXN-CPD-18362/S-ADENOSYLMETHIONINE//CPD-18406/ADENOSYL-HOMO-CYS/PROTON.67.
  2.1.1.79-RXN-CPD-18367/S-ADENOSYLMETHIONINE//CPD-18371/ADENOSYL-HOMO-CYS/PROTON.67.
  2.1.1.79-RXN-CPD-18369/S-ADENOSYLMETHIONINE//CPD-18372/ADENOSYL-HOMO-CYS/PROTON.67.
  2.1.1.79-RXN-CPD-18392/S-ADENOSYLMETHIONINE//CPD-18405/ADENOSYL-HOMO-CYS/PROTON.67.
  2.1.1.79-RXN-CPD-18403/S-ADENOSYLMETHIONINE//CPD-18404/ADENOSYL-HOMO-CYS/PROTON.67.
  2.3.1.157-RXN


In [7]:
# Listener target arrays. t=0 is empty (the process hasn't run yet) — sample t=1.
print(f"Number of timesteps:       {len(fba['target_kinetic_fluxes'])}")
print(f"target_kinetic_fluxes[1]:  shape {np.array(fba['target_kinetic_fluxes'][1]).shape}     (n_kin,)")
print(f"target_kinetic_bounds[1]:  shape {np.array(fba['target_kinetic_bounds'][1]).shape}  (n_kin, 2) = [lower, upper]")
print(f"target_homeostatic_dmdt[1]: shape {np.array(fba['target_homeostatic_dmdt'][1]).shape}     (n_homeo,)")


Number of timesteps:       601
target_kinetic_fluxes[1]:  shape (416,)     (n_kin,)
target_kinetic_bounds[1]:  shape (416, 2)  (n_kin, 2) = [lower, upper]
target_homeostatic_dmdt[1]: shape (172,)     (n_homeo,)


In [8]:
# metabolism_redux objective weights.
# Note: `efficiency` and `homeostatic` keys from the classic API are silently
# ignored — redux has no `efficiency` term and applies the homeostatic loss
# unconditionally. `kinetics_in_range` lightly penalizes fluxes inside the
# [lower, upper] kinetic band; `kinetics` heavily penalizes fluxes outside it.
objective_weights = {
    "secretion": 1e-3,
    "kinetics": 1e-7,
    "kinetics_in_range": 1e-2,
}

# Simulate adding a new reaction (gene addition proxy).
add_reaction = {
    'TEMP-REACTION-1': {
        'APO-CITRATE-LYASE[c]': 1,
    },
    'TEMP-REACTION-2': {
        'CIT[c]': -1,
        'CITRATE-LYASE[c]': -1,
    },
}

# force_reaction is not supported in metabolism_redux. For the equivalent
# pattern (force flux through a reaction), use:
#   add_kinetic = {'RXN-ID': flux_value}
#   remove_reaction = ['RXN-ID']   # prevent opposing-direction futile cycle
# force_reaction = ['OXALODECARB-RXN']

# Swap carbon source: citrate in, glucose out
uptake_addition = set(['CIT[p]'])
uptake_removal = set(['GLC[p]'])

# Knockout of a downstream citrate degradation reaction
remove_reaction = ['CITTRANS-RXN']


In [9]:
result = test_NetworkFlowModel(
    objective_weights,
    metabolism=metabolism,
    fba=fba,
    bulk=bulk,
    uptake_addition=uptake_addition,
    add_reaction=add_reaction,
    remove_reaction=remove_reaction,
)

[oofv, solution_flux, test_reaction_names, S_new, metabolites_new, kinetics_new] = result


In [10]:
# Solver diagnostic: did it converge, and how many reactions carry nonzero flux?
print(f"Objective value: {oofv:,.4g}")
n_active = int((np.abs(solution_flux) > 0).sum())
print(f"Reactions carrying nonzero flux: {n_active:,} of {len(solution_flux):,}")


Objective value: 59.52
Reactions carrying nonzero flux: 639 of 9,422


In [11]:
sim_flux = pd.DataFrame({
    'flux': solution_flux,
    'is_new': [
        'TEMP' if id in add_reaction.keys()
        else 'Original Reactions'
            for id in test_reaction_names
    ]
}, index=test_reaction_names)

This concludes running the stand-alone FBA. Now we can do various analysis on this output

In [12]:
# Look at fluxes of reactions involving a certain metabolite of interest.
met_of_interest = ['CITRATE-LYASE[c]', 'OXALACETIC_ACID[c]', 'CIT[c]']
S_new = pd.DataFrame(S_new, index=metabolites_new, columns=test_reaction_names)

S_met, rxns = get_subset_S(S_new, met_of_interest)

rxn_flux = sim_flux.loc[rxns]
rxn_flux['kinetic'] = [
    kinetics_new.loc[r, "target"] if r in kinetics_new.index else None
    for r in rxn_flux.index
]
rxn_flux


,flux,is_new,kinetic
2-METHYLCITRATE-SYNTHASE-RXN,0.000000e+00,Original Reactions,NaN
2.3.1.49-RXN,0.000000e+00,Original Reactions,NaN
ACONITATEDEHYDR-RXN,2.188134e+05,Original Reactions,NaN
ACONITATEDEHYDR-RXN (reverse),0.000000e+00,Original Reactions,NaN
ASPAMINOTRANS-RXN (reverse),0.000000e+00,Original Reactions,NaN
ASPAMINOTRANS-RXN__ASPAMINOTRANS-DIMER,3.047935e+05,Original Reactions,8.062939e+05
ASPAMINOTRANS-RXN__ASPAMINOTRANS-DIMER (reverse),2.701353e+05,Original Reactions,2.415775e+06
ASPAMINOTRANS-RXN__TYRB-DIMER,2.307112e+03,Original Reactions,8.738283e+03
CITC-RXN,0.000000e+00,Original Reactions,NaN
CITLY-RXN,0.000000e+00,Original Reactions,NaN


#### Bar plot of fluxes of reactions involving the metabolite of interest

In [13]:
# Top 10 reactions by absolute flux
top10_flux = (
    sim_flux.assign(abs_flux=sim_flux["flux"].abs())
    .sort_values("abs_flux", ascending=False)
    .head(10)
    .copy()
)

# Keep labels in plotting order (largest first)
top10_flux = top10_flux.sort_values("abs_flux", ascending=True)

# Symmetric color range so RdBu's diverging midpoint sits at zero regardless
# of whether the selected reactions are all-positive, all-negative, or mixed.
vmax = float(top10_flux["flux"].abs().max())

fig = px.bar(
    top10_flux,
    x="abs_flux",
    y=top10_flux.index,
    orientation="h",
    color="flux",  # preserves sign information
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0,
    range_color=[-vmax, vmax],
    labels={"abs_flux": "|Flux|", "y": "Reaction", "flux": "Signed flux"},
    title="Top 10 Reactions by Flux Magnitude",
)

fig.update_layout(
    yaxis_title="Reaction",
    xaxis_title="Absolute flux",
    template="plotly_white",
    height=500,
)
fig.update_xaxes(type="log")
fig.show()


### Scatterplot of the reactions with kinetic target

In [14]:
# Keep only reactions with a defined kinetic target
kinetic_compare = (
    rxn_flux[["flux", "kinetic"]]
    .dropna(subset=["kinetic"])
    .rename(columns={"flux": "simulated_flux", "kinetic": "target_kinetic_flux"})
    .copy()
)

# If there are no kinetic targets in this subset, this will print and skip plotting
if kinetic_compare.empty:
    print("No reactions in rxn_flux have kinetic targets.")
else:
    # Build symmetric range for y=x reference line
    min_val = min(
        kinetic_compare["simulated_flux"].min(),
        kinetic_compare["target_kinetic_flux"].min(),
    )
    max_val = max(
        kinetic_compare["simulated_flux"].max(),
        kinetic_compare["target_kinetic_flux"].max(),
    )

    fig = px.scatter(
        kinetic_compare.reset_index().rename(columns={"index": "reaction"}),
        x="target_kinetic_flux",
        y="simulated_flux",
        hover_data=["reaction"],
        title="Simulated vs Target Kinetic Flux (Reactions with Kinetic Targets)",
        labels={
            "target_kinetic_flux": "Target kinetic flux (count/s)",
            "simulated_flux": "Simulated flux (count/s)",
        },
        template="plotly_white",
    )

    # Add y = x line (perfect agreement)
    fig.add_shape(
        type="line",
        x0=min_val,
        y0=min_val,
        x1=max_val,
        y1=max_val,
        line=dict(color="gray", dash="dash"),
    )

    fig.update_layout(height=500)
    fig.update_xaxes(type="log")
    fig.update_yaxes(type="log")
    fig.show()